# StatMod smoke test: prototype lab notebook

**Test content only.** This notebook checks the lab round trip (GitHub, Colab, download) for the
module 5PAM2024 Statistical Modelling. It contains no teaching material and no answers.

- **Data:** two small subsamples of the UCI Combined Cycle Power Plant dataset (Tüfekci and Kaya;
  CC BY 4.0, DOI 10.24432/C5002N), labelled as smoke-test fixtures. `temp_band` and
  `humidity_band` are derived from `AT` and `RH` for testing formulas.
- **Run the cells from the top.** The exercise cell stops with a message until it is completed;
  that is by design. The dataset-entry cell needs the test ID given in the checklist.

In [ ]:
# VERSIONS: run this cell first. It records the software this notebook ran with.
import datetime
import importlib.util
import platform
import sys
from importlib import metadata

TESTED_WITH = {  # versions the local smoke test passed with (statmod-colab, 8 October 2026)
    "numpy": "2.1.3",
    "pandas": "2.2.3",
    "scipy": "1.16.3",
    "statsmodels": "0.15.0",
    "matplotlib": "3.10.0",
    "patsy": "1.0.3",
    "formulaic": "1.2.2",
    "seaborn": "0.13.2",
}
try:  # find_spec raises, rather than returning None, where there is no google package at all
    IN_COLAB = importlib.util.find_spec("google.colab") is not None
except ModuleNotFoundError:
    IN_COLAB = False
print(f"Run at        {datetime.datetime.now(datetime.timezone.utc):%Y-%m-%d %H:%M} UTC")
print(f"Python        {platform.python_version()}  ({sys.executable})")
print(f"Platform      {platform.platform()}")
print(f"In Colab      {IN_COLAB}")
for package, tested in TESTED_WITH.items():
    try:
        version = metadata.version(package)
    except metadata.PackageNotFoundError:
        version = "not installed"
    note = "" if version == tested else f"   (tested with {tested})"
    print(f"{package:<13} {version}{note}")

In [ ]:
# SET-UP: run this cell before any other. It defines the helpers for loading and checking data.
import hashlib
import io
import os
import urllib.request
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf

pd.set_option("display.width", 120)  # printed tables wrap less

BRANCH = "smoke-test"
DATA_URL = os.environ.get(  # the variable lets the local smoke test point elsewhere
    "STATMOD_DATA_URL", f"https://raw.githubusercontent.com/uh-pam/statmod/{BRANCH}/data/"
)
LOCAL_DATA = Path("data")  # offline copy beside the notebook: the lab-PC fallback

DATASETS = {  # every file this notebook may load, with its published checksum
    "SHARED": {
        "file": "smoke_shared.csv",
        "name": "smoke-test shared data (CCPP subsample, 200 rows)",
        "sha256": "c4617c32b247d19444826d9ae84b9c657b97ded172e8286d57d49748cadbc93d",
    },
    "T07": {
        "file": "smoke_T07.csv",
        "name": "smoke-test individual data T07 (CCPP subsample, 120 rows)",
        "sha256": "df8c1fd9e4dc0f22711aa532cce44c93afbf79bc79ad81467dfae1e46ce03f79",
    },
}
ASSESSED_IDS = ("T07",)  # the IDs the dataset-entry cell accepts


class LabStop(Exception):
    """An expected stop: shown as one plain message, without a long traceback."""

    def _render_traceback_(self):
        return [f"{type(self).__name__}: {self}"]


class DatasetError(LabStop):
    pass


class ExerciseNotDone(LabStop):
    pass


def fetch_bytes(filename):
    """Download a data file; if that fails, say so and use the local copy."""
    url = DATA_URL + filename
    try:
        with urllib.request.urlopen(url, timeout=15) as response:
            return response.read(), url
    except Exception as exc:  # no network, unknown host, HTTP error, timeout
        local = LOCAL_DATA / filename
        print(f"Could not download {url} ({getattr(exc, 'reason', exc)}).")
        if local.is_file():
            print(f"Using the local copy instead: {local}")
            return local.read_bytes(), str(local)
        raise DatasetError(
            f"{filename} could not be downloaded and there is no local copy at {local}. "
            "Check the internet connection and run the cell again; if it still fails, "
            "tell the supervisor."
        ) from None


def load_dataset(dataset_id):
    """Load a dataset, verify its checksum, and echo its name, size and checksum."""
    entry = DATASETS[dataset_id]
    raw, source = fetch_bytes(entry["file"])
    digest = hashlib.sha256(raw).hexdigest()
    if digest != entry["sha256"]:
        raise DatasetError(
            f"{entry['file']} from {source} is not the published version "
            f"(sha256 {digest[:12]}..., expected {entry['sha256'][:12]}...). "
            "Tell the supervisor."
        )
    frame = pd.read_csv(io.BytesIO(raw))
    print(f"Loaded {dataset_id}: {entry['name']}")
    print(f"  source  {source}")
    print(f"  size    {frame.shape[0]} rows x {frame.shape[1]} columns, {len(raw)} bytes")
    print(f"  sha256  {digest}")
    return frame


_ASSESSED = {}  # filled only by enter_dataset(); emptied whenever an entry is attempted


def enter_dataset(typed_id):
    """Validate the ID typed in the dataset-entry cell and load that dataset."""
    _ASSESSED.clear()  # a failed or changed entry never leaves an older dataset in use
    dataset_id = str(typed_id).strip().upper()
    if not dataset_id:
        raise DatasetError(
            "No dataset ID yet. Type the ID you were given between the quotes, "
            "then run this cell again."
        )
    if dataset_id not in ASSESSED_IDS:
        raise DatasetError(
            f"{typed_id!r} is not a dataset ID for this lab. Check the ID you were given "
            "(a letter followed by two digits), correct it between the quotes and run this "
            "cell again."
        )
    frame = load_dataset(dataset_id)
    _ASSESSED.update(id=dataset_id, sha256=DATASETS[dataset_id]["sha256"], data=frame)


def require_dataset():
    """Return a fresh copy of the validated dataset, or stop with a message."""
    if "data" not in _ASSESSED:
        raise DatasetError(
            "No valid dataset is loaded. Run the DATASET ENTRY cell with your dataset ID first."
        )
    print(f"Using dataset {_ASSESSED['id']} (sha256 {_ASSESSED['sha256'][:12]}...)")
    frame = _ASSESSED["data"].copy()
    frame.attrs["dataset_id"] = _ASSESSED["id"]
    return frame


def check_done(name, value):
    """Stop with a message while an exercise still holds its placeholder (...)."""
    if value is ...:
        raise ExerciseNotDone(
            f"Exercise not finished: replace the ... in `{name} = ...` with your code, "
            "then run this cell again. (This message is expected until you do.)"
        )


print("Set-up complete.")

In [ ]:
# SEED: fixes every random step in this notebook, so a re-run gives the same results.
SEED = 20261008
rng = np.random.default_rng(SEED)

## Formative steps (smoke test)

These steps load the shared dataset automatically.

In [ ]:
shared = load_dataset("SHARED")
shared.sample(5, random_state=rng)

In [ ]:
# OLS with C() terms, an interaction and a numeric covariate; ANOVA tables of Types I, II and III.
fit_t = smf.ols("PE ~ C(temp_band) * C(humidity_band) + V", data=shared).fit()
print(fit_t.params.round(4))
print("\nType I\n", sm.stats.anova_lm(fit_t, typ=1).round(4))
print("\nType II\n", sm.stats.anova_lm(fit_t, typ=2).round(4))
fit_s = smf.ols("PE ~ C(temp_band, Sum) * C(humidity_band, Sum) + V", data=shared).fit()
print("\nType III (sum-to-zero coding)\n", sm.stats.anova_lm(fit_s, typ=3).round(4))

In [ ]:
# Influence measures: hat values, studentised residuals and Cook's distance.
influence = fit_t.get_influence()
measures = pd.DataFrame(
    {
        "hat": influence.hat_matrix_diag,
        "studentised": influence.resid_studentized_external,
        "cooks_d": influence.cooks_distance[0],
    },
    index=shared["ccpp_row"],
)
measures.sort_values("cooks_d", ascending=False).head().round(4)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 3.5))
axes[0].scatter(fit_t.fittedvalues, fit_t.resid, s=10)
axes[0].axhline(0, color="grey", linewidth=1)
axes[0].set(xlabel="Fitted PE (MW)", ylabel="Residual (MW)", title="Residuals against fitted")
sm.qqplot(influence.resid_studentized_external, line="45", ax=axes[1])
axes[1].set_title("Normal Q-Q plot of studentised residuals")
fig.tight_layout()
plt.show()

### Exercise E1 (smoke test)

Fit `PE ~ AT` to `shared` with `smf.ols` and store the fitted model in `fit_e1`. Until you do,
the cell stops with a message: that is the designed behaviour being tested.

In [ ]:
fit_e1 = ...  # replace ... with your code
check_done("fit_e1", fit_e1)
print(f"E1 recorded: {fit_e1.model.formula}, n = {int(fit_e1.nobs)}")

In [ ]:
# STRETCH S1 (optional, smoke test): replace ... with any extra plot of `shared`.
# Left untouched, this cell skips itself without an error.
stretch_s1 = ...
if stretch_s1 is ...:
    print("Stretch S1 not attempted: skipped.")

## Assessed part (smoke test)

### Dataset entry

Type the dataset ID you were given in the next cell, between the quotes, and run it. The cells
after it stop with a message until a valid dataset is loaded.

In [ ]:
# ============================== DATASET ENTRY ==============================
# Type the dataset ID you were given between the quotes, then run this cell.
DATASET_ID = ""
# ===========================================================================
enter_dataset(DATASET_ID)

In [ ]:
data = require_dataset()  # NameError here? Run the cells from the top first.
fit_a = smf.ols("PE ~ C(temp_band, Sum) * C(humidity_band, Sum) + V", data=data).fit()
print(sm.stats.anova_lm(fit_a, typ=3).round(4))

In [ ]:
data = require_dataset()  # NameError here? Run the cells from the top first.
fit_a = smf.ols("PE ~ C(temp_band, Sum) * C(humidity_band, Sum) + V", data=data).fit()
influence_a = fit_a.get_influence()
cooks_a = pd.Series(influence_a.cooks_distance[0], index=data["ccpp_row"], name="cooks_d")
print("Largest Cook's distances:\n", cooks_a.nlargest(3).round(4))
fig, ax = plt.subplots(figsize=(5, 3.5))
ax.scatter(influence_a.hat_matrix_diag, influence_a.resid_studentized_external, s=12)
ax.set(
    xlabel="Leverage (hat value)",
    ylabel="Studentised residual",
    title=f"Dataset {data.attrs['dataset_id']}",
)
plt.show()

End of the smoke-test notebook. Download it with its outputs (see the checklist).